In [1]:
import sys
sys.path.append("..")

In [2]:
import numpy as np
import pandas as pd
import os
import plotly.express as px
from sklearn.model_selection import ParameterGrid

from src.data import *
from src.utils import *
from src.model import *
from src.recourse import *

In [3]:
from sklearn.linear_model import LogisticRegression
import plotly.express as px

In [4]:
# Linf
# Grid Search

def getStats(x: np.ndarray, theta: np.ndarray, x0: np.ndarray, lamb):
    return np.log(1 + np.exp(-(x @ theta))) + \
        (lamb * (np.linalg.norm(x0 - x, ord=1)))

def calThetaAdv_l1(xP: np.ndarray, theta0: np.ndarray, alpha):
    # xP has bias
    thetaP = theta0.copy()
    i = np.argmax(np.abs(xP))
    thetaP[i] -= (alpha * np.sign(xP[i]))

    return thetaP

def calThetaAdv_linf(xP: np.ndarray, weights: np.ndarray, bias, alpha):
    # xP does not have bias
    weights_adv = weights - (alpha * np.sign(xP))

    for i in range(len(xP)):
        if np.sign(xP[i]) == 0:
            weights_adv[i] = weights_adv[i] - (alpha * np.sign(weights_adv[i]))
    bias_adv = bias - alpha

    return np.hstack((weights_adv, bias_adv))

def getAllPossibleX(x0: np.ndarray, length=6, step_size=0.1):
    # x0 don't have bias
    x0 = np.hstack((x0, np.array([1])))
    d = [[(length - 1) * -abs(x), length * abs(x)] for x in x0]
    d[x0.size- 1][0] = 1
    d[x0.size- 1][1] = 1

    delta_x = [np.arange(d[i][0], d[i][1] + step_size/2, step_size) for i in range(len(x0))]
    X = np.array(np.meshgrid(*delta_x)).T.reshape(-1, x0.shape[0])
    return np.round(X, decimals=5)

def searchGrid_linf(X, x0_withBias, theta0, bias0, alpha, lamb):
    Js = np.apply_along_axis(lambda x : getStats(x, calThetaAdv_linf(x[:-1], theta0, bias0, alpha), x0_withBias, lamb), arr=X, axis=1)
    Js_min_i = np.argmin(Js)
    xR_new_GS = X[Js_min_i]

    return xR_new_GS[:-1]

In [5]:
# def calThetaAdv_linf(xP: np.ndarray, weights: np.ndarray, bias, alpha):
#     # xP does not have bias
#     weights_adv = weights - (alpha * np.sign(xP))

#     for i in range(len(xP)):
#         if np.sign(xP[i]) == 0:
#             weights_adv[i] = weights_adv[i] - (alpha * np.sign(weights_adv[i]))
#     bias_adv = bias - alpha

#     # return np.concat((weights_adv, bias_adv))
#     return (weights_adv, bias_adv)

In [6]:
def getTheta0AndBias0 (ret):
    try:
        theta0 = ret['theta_0']['out.weight'][0].numpy().astype(np.float64)
        bias0 = ret['theta_0']['out.bias'].numpy().astype(np.float64) 
    except TypeError:
        try: 
            theta0 = ret['theta_0'][0][0][0][0].numpy().astype(np.float64)
            bias0 = ret['theta_0'][0][0][1].numpy().astype(np.float64)
        except IndexError:
            theta0 = ret['theta_0'][0][0][0].numpy().astype(np.float64)
            bias0 = ret['theta_0'][0][1].numpy().astype(np.float64)
    
    return theta0, bias0

### RBR Files

In [14]:
file_path = "../results/cost_validity_latest/lr_synthesis_alg1_lamb0.2_new.pickle"
ret = pd.read_pickle(file_path)

x0 = ret['x_0'][6][0]
x0_withBias = np.hstack((x0, np.array([1])))
xR_old = ret['x_r'][6][0]
xR_old_withBias = np.hstack((xR_old, np.array([1])))

# theta0 = ret['theta_0'][0][0][0][0].numpy().astype(np.float64)
# bias0 = ret['theta_0'][0][0][1].numpy().astype(np.float64) 
theta0, bias0 = getTheta0AndBias0(ret)
# divider = np.linalg.norm(theta0)
# theta0 = ret['theta_0'][0][0][0][0].numpy().astype(np.float64) / divider
# bias0 = ret['theta_0'][0][0][1].numpy().astype(np.float64) /divider
theta0_withBias = np.hstack((theta0, bias0))

alpha = 0.7
lamb = 0.2

In [15]:
lInfR = LARRecourse(weights=theta0, bias=bias0, alpha=alpha , lamb=lamb)
xR_new = lInfR.get_recourse(x0)
xR_new_withBias = np.hstack((xR_new, np.array([1])))

In [24]:
X = getAllPossibleX(x0, length=3, step_size=0.02)

In [25]:
# Js = np.apply_along_axis(lambda x : getStats(x, calThetaAdv_linf(x[:-1], theta0, bias0, alpha), x0_withBias, lamb), arr=X, axis=1)
# Js_min_i = np.argmin(Js)
# xR_new_GS = X[Js_min_i]

xR_new_GS = searchGrid_linf(X, x0_withBias, theta0, bias0, alpha, lamb)

In [28]:
J_xR_old = getStats(xR_old_withBias, calThetaAdv_linf(xR_old, theta0, bias0, alpha), x0_withBias, lamb)
J_xR_new = getStats(xR_new_withBias, calThetaAdv_linf(xR_new, theta0, bias0, alpha), x0_withBias, lamb)
J_xR_new_GS = getStats(np.hstack((xR_new_GS, np.array([1]))), calThetaAdv_linf(xR_new_GS, theta0, bias0, alpha), x0_withBias, lamb)

print(f"X0 : {x0}")
print(f"Theta0 : {theta0_withBias}")
print(f"Alpha: {alpha}")
print(f"Lamb: {lamb}")
print(f"XR Old Linf: {xR_old}, J : {J_xR_old}")
print(f"XR New Linf: {xR_new}, J : {J_xR_new}")
print(f"XR GS: {xR_new_GS}, J : {J_xR_new_GS}")

X0 : [1.8468858  5.56784421]
Theta0 : [ 1.02545357 -0.17008451 -0.95659643]
Alpha: 0.7
Lamb: 0.2
XR Old Linf: [3.6570955 0.       ], J : 2.428894875433304
XR New Linf: [3.6570955 0.       ], J : 2.428894875433304
XR GS: [3.66623 0.00431], J : 2.4303374626388607


In [51]:
# Newly Generated Files

file_path = "../results/cost_validity_latest/lr_synthesis_alg1_lamb0.3_new.pickle"
ret = pd.read_pickle(file_path)

x0 = ret['x_0'][0][98]
x0_withBias = np.hstack((x0, np.array([1])))
xR_old = ret['x_r'][0][98]
xR_old_withBias = np.hstack((xR_old, np.array([1])))

theta0 = ret['theta_0']['out.weight'][0].numpy().astype(np.float64)
bias0 = ret['theta_0']['out.bias'].numpy().astype(np.float64)
# divider = np.linalg.norm(theta0, 2)
# theta0 = theta0 / divider
# bias0 = bias0 /divider
theta0_withBias = np.hstack((theta0, bias0))

alpha = 0.02
lamb = 0.3

tmp = ret['x_r']

In [52]:
lInfR = LARRecourse(weights=theta0, bias=bias0, alpha=alpha , lamb=lamb)
xR_new = lInfR.get_recourse(x0)
xR_new_withBias = np.hstack((xR_new, np.array([1])))

In [58]:
J_xR_old = getStats(xR_old_withBias, calThetaAdv_linf(xR_old, theta0, bias0, alpha), x0_withBias, lamb)
J_xR_new = getStats(xR_new_withBias, calThetaAdv_linf(xR_new, theta0, bias0, alpha), x0_withBias, lamb)

print(f"X0 : {x0}")
print(f"Theta0 : {theta0_withBias}")
print(f"Alpha: {alpha}")
print(f"Lamb: {lamb}")
print(f"XR Old Linf: {xR_old}, J : {J_xR_old}")
print(f"XR New Linf: {xR_new}, J : {J_xR_new}")

X0 : [1.90852233 5.06407279]
Theta0 : [ 0.5587728  -0.30975455  0.46848744]
Alpha: 0.02
Lamb: 0.3
XR Old Linf: [1.90852233 5.06407279], J : 0.7943803330939633
XR New Linf: [1.90852233 5.06407279], J : 0.7943803330939633


In [ ]:
# Newly Generated Files

file_path = "../results/cost_validity_latest/lr_sba_alg1_lamb0.2.pickle"
ret = pd.read_pickle(file_path)

x0 = ret['x_0'][5][5]
x0_withBias = np.hstack((x0, np.array([1])))
xR_old = ret['x_r'][5][5]
xR_old_withBias = np.hstack((xR_old, np.array([1])))

try:
    theta0 = ret['theta_0']['out.weight'][0].numpy().astype(np.float64)
    bias0 = ret['theta_0']['out.bias'].numpy().astype(np.float64) 
except TypeError:
    try: 
        theta0 = ret['theta_0'][0][0][0][0].numpy().astype(np.float64)
        bias0 = ret['theta_0'][0][0][1].numpy().astype(np.float64)
    except IndexError:
        theta0 = ret['theta_0'][0][0][0].numpy().astype(np.float64)
        bias0 = ret['theta_0'][0][1].numpy().astype(np.float64)
# divider = np.linalg.norm(theta0, 2)
# theta0 = theta0 / divider
# bias0 = bias0 /divider
theta0_withBias = np.hstack((theta0, bias0))

alpha = 0.1
lamb = 0.2

tmp = ret['x_r']

lInfR = LARRecourse(weights=theta0, bias=bias0, alpha=alpha , lamb=lamb)
xR_new = lInfR.get_recourse(x0)
xR_new_withBias = np.hstack((xR_new, np.array([1])))

J_xR_old = getStats(xR_old_withBias, calThetaAdv_linf(xR_old, theta0, bias0, alpha), x0_withBias, lamb)
J_xR_new = getStats(xR_new_withBias, calThetaAdv_linf(xR_new, theta0, bias0, alpha), x0_withBias, lamb)

print(f"X0 : {x0}")
print(f"Theta0 : {theta0_withBias}")
print(f"Alpha: {alpha}")
print(f"Lamb: {lamb}")
print(f"XR Old Linf: {xR_old}, J : {J_xR_old}")
print(f"XR New Linf: {xR_new}, J : {J_xR_new}")

X0 : [-0.26558718 -0.06867307  0.49755874 -0.4268808  -0.03345935  1.76401877
  1.62572474 -0.27119077 -0.30590044  0.71882223  1.20519906  0.
  1.          0.        ]
Theta0 : [ 0.77824163 -0.27275857  0.11780936 -0.12219541 -0.05773614  0.21237242
 -0.06366055  0.29351866  0.24285752 -0.55687368 -0.37835842  0.67323387
  0.42163563  0.08462106 -0.14200011]
Alpha: 0.1
Lamb: 0.2
XR Old Linf: [ 0.87201303 -0.06867307  0.49755874 -0.4268808  -0.03345935  1.76401877
  1.62572474 -0.27119077 -0.30590044  0.          1.20519906  0.
  1.          0.        ], J : 1.1455240317853597
XR New Linf: [ 3.08372231 -0.06867307  0.49755874 -0.4268808  -0.03345935  1.76401877
  1.62572474 -0.27119077 -0.30590044  0.71882223  1.20519906  0.
  1.          0.        ], J : 1.0192493983840216


In [8]:
# Checking whether validity is non-monotonic

# alphas = np.linspace(0.02, 1,50)
# xRs = np.empty((len(alphas), len(x0)))

# for i, alpha in enumerate(alphas):

#     tmpL1 = LARRecourse(weights=theta0, bias=bias0, alpha=alpha , lamb=0.2)
#     xRs[i] = tmpL1.get_recourse(x0)

#     clf = LogisticRegression()
#     weightsR, biasR = calThetaAdv_linf(xRs[i], theta0, bias0, alpha)
#     clf.coef_ = weightsR.reshape(1,-1)
#     clf.intercept_ = biasR
#     clf.classes_ = np.array([0,1])
#     print(f"-----------------------------------") 
#     print(f"alpha {alpha}")
#     print(f"x0: {x0}")
#     print(f"XR: {xRs[i]}")
#     print(f"ThetaP: {weightsR}, {biasR}")   
#     print(f"Prob: {clf.predict_proba(xRs[i].reshape(1,-1))[0,1]}")

#     # J = RecourseCost(x_0=x0, lamb=0.4)
#     # print(J.eval(x=xRs[i], weights= weightsR, bias=biasR, breakdown=True))

### Our Files

In [14]:
file_path = "../results/recourse/lr_sba_L1PSD_1.4_0.1_4.pkl"
ret = pd.read_pickle(file_path)

In [16]:
num_i = 3
l1 = L1Recourse(weights= ret.loc[num_i, 'theta_0'][:-1], bias= ret.loc[num_i, 'theta_0'][[-1]], alpha= ret.loc[num_i, 'alpha'], lamb= ret.loc[num_i, 'lambda'])
xR = l1.get_recourse(ret.loc[num_i, 'x_0'])

In [17]:
# Saved File
xP_old_withB = np.hstack((ret['x_r'][num_i], [1]))
# New
xP_new_withB = np.hstack((xR, [1]))

In [18]:
if "L1" in l1.name:
    thetaP_old = calThetaAdv_l1(xP_old_withB, ret.loc[num_i, 'theta_0'], ret.loc[num_i, 'alpha'])
    thetaP_new = calThetaAdv_l1(xP_new_withB, ret.loc[num_i, 'theta_0'], ret.loc[num_i, 'alpha'])
else:
    thetaP_old = calThetaAdv_linf(xP_old_withB[:-1], ret.loc[num_i, 'theta_0'][:-1], ret.loc[num_i, 'theta_0'][[-1]], ret.loc[num_i, 'alpha'])
    thetaP_new = calThetaAdv_linf(xP_new_withB[:-1], ret.loc[num_i, 'theta_0'][:-1], ret.loc[num_i, 'theta_0'][[-1]], ret.loc[num_i, 'alpha'])


j_old = getStats(xP_old_withB, thetaP_old, np.hstack((ret.loc[num_i, 'x_0'], [1])), ret.loc[num_i, 'lambda'])
j_new = getStats(xP_new_withB, thetaP_new, np.hstack((ret.loc[num_i, 'x_0'], [1])), ret.loc[num_i, 'lambda'])

In [20]:
print(f"{l1.name}")
print(f"xP (old): {xP_old_withB}")
print(f"J (old): {j_old}")
print(f"xP (new): {xP_new_withB.round(4)}")
print(f"J (new): {j_new}")

L1PSD
xP (old): [-0.7294 -0.7562  0.3464  0.3638 -1.299  -0.2563 -0.4234 -0.0863 -0.1726
 -0.1693  0.3015  0.7314  0.3172 -0.7112  0.5705 -0.6031 -0.6716 -1.299
 -1.299  -1.0631 -0.0231  0.0284  0.036  -0.0568  0.9613  1.    ]
J (old): 3.861181012242064
xP (new): [-7.2990e-01 -8.2830e-01  3.1850e-01  3.5590e-01 -1.2994e+00 -2.2750e-01
 -3.7770e-01 -8.2400e-02 -1.3950e-01 -1.9130e-01  3.4570e-01  7.1220e-01
  2.8280e-01 -6.8020e-01  5.2610e-01 -6.6340e-01 -6.8240e-01 -1.1660e+00
 -1.2994e+00 -1.0580e+00 -1.0000e-03  3.0000e-04  2.0000e-04  6.0000e-04
  1.0008e+00  1.0000e+00]
J (new): 2.7443887252054746
